In [1]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2:1b")

c:\Users\dev-euna\Desktop\AI_AGENT_STUDY\langchain-basic\langchain-basic\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 명시적인 지시사항이 포함된 프롬프트 템플릿 정의
prompt_template = PromptTemplate(
    template="What is the capital of {country}? Return the name of the city only",
    input_variables=["country"],
)
# 템플릿에 변수 값을 주입
prompt = prompt_template.invoke({"country": "France"})

print(prompt)

# LLM에 프롬프트 전달
ai_message = llm.invoke(prompt_template.invoke({"country": "France"}))

print(ai_message)

# 문자열 출력 파서를 사용하여 응답을 단순 문자열로 변환
output_parser = StrOutputParser()

answer = output_parser.invoke(llm.invoke(prompt_template.invoke({"country": "France"})))

text='What is the capital of France? Return the name of the city only'
content='Paris' additional_kwargs={} response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-03-09T09:25:47.8639008Z', 'done': True, 'done_reason': 'stop', 'total_duration': 660669500, 'load_duration': 301875200, 'prompt_eval_count': 39, 'prompt_eval_duration': 301003600, 'eval_count': 2, 'eval_duration': 54148500, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'} id='lc_run--019cd1ea-ba41-7040-b5e3-0269c22dafd6-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 39, 'output_tokens': 2, 'total_tokens': 41}


In [3]:
ai_message

AIMessage(content='Paris', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-03-09T09:25:47.8639008Z', 'done': True, 'done_reason': 'stop', 'total_duration': 660669500, 'load_duration': 301875200, 'prompt_eval_count': 39, 'prompt_eval_duration': 301003600, 'eval_count': 2, 'eval_duration': 54148500, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--019cd1ea-ba41-7040-b5e3-0269c22dafd6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 39, 'output_tokens': 2, 'total_tokens': 41})

In [4]:
answer

'Paris'

In [5]:
from langchain_core.output_parsers import JsonOutputParser

country_detail_prompt = PromptTemplate(
    template="""Give following information about {country}:" \
    - Capital
    - Population
    - Language
    - Currency

    return it in JSON format. and return the JSON dictionary only
    """,
    input_variables=["country"],
)

country_detail_prompt.invoke({"country": "France"})

output_parser = JsonOutputParser()

json_ai_message = llm.invoke(prompt_template.invoke({"country": "France"}))
output_parser.invoke(json_ai_message)


OutputParserException: Invalid json output: Paris
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 

In [6]:
type(json_ai_message)

langchain_core.messages.ai.AIMessage

In [9]:
from pydantic import BaseModel, Field

# 국가 정보를 담을 Pydantic 모델 정의
class CountryDetail(BaseModel):
    capital: str = Field(description="The capital of the country")
    population: int = Field(description="The population of the country")
    language: str = Field(description="The language of the country")
    currency: str = Field(description="The currency of the country")


# LLM에 구조화된 출력 형식 지정
structued_llm = llm.with_structured_output(CountryDetail)

country_detail = structued_llm.invoke(country_detail_prompt.invoke({"country": "France"}))
country_detail

CountryDetail(capital='Paris', population=67, language='French', currency='Euro')